# Direct Preference Optimization (DPO) Fine-Tuning Workshop

In [1]:
import os
import torch
import warnings
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig
from trl import DPOTrainer, DPOConfig

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    else torch.float16
)
print(f"Using device: cuda | Dtype: {compute_dtype}")

Using device: cuda | Dtype: torch.bfloat16


In [2]:
MODEL_ID = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=compute_dtype,
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
print("Model and PEFT adapters loaded successfully.")

Model and PEFT adapters loaded successfully.


In [3]:
dataset = load_dataset("argilla/distilabel-intel-orca-dpo-pairs", split="train")
shuffled = dataset.shuffle(seed=42)
train_ds = shuffled.select(range(250))
eval_ds = shuffled.select(range(250, 300))


def format_dpo_example(example):
    messages = [{"role": "user", "content": example["input"]}]
    prompt_str = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    chosen_str = example["chosen"].strip() + tokenizer.eos_token
    rejected_str = example["rejected"].strip() + tokenizer.eos_token
    return {
        "prompt": prompt_str,
        "chosen": chosen_str,
        "rejected": rejected_str,
    }


train_mapped = train_ds.map(format_dpo_example, remove_columns=train_ds.column_names)
eval_mapped = eval_ds.map(format_dpo_example, remove_columns=eval_ds.column_names)
print("Prompt Preview:
", train_mapped[0]["prompt"])
print("Chosen Preview:
", train_mapped[0]["chosen"])
print("Rejected Preview:
", train_mapped[0]["rejected"])

In [4]:
test_prompts = [
    "Explain why the sky is blue in one concise sentence.",
    "What is a metric ton?",
    "Why did the Roman Empire fall? Explain the primary contributing factors.",
    "What is the difference between existentialism and nihilism?",
]

base_responses = {}
for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            pad_token_id=tokenizer.pad_token_id,
        )
    resp = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    ).strip()
    base_responses[prompt] = resp
    print(f"Prompt: {prompt}
Base Response: {resp}
" + "-" * 50)

Prompt: Explain why the sky is blue in one concise sentence.
Base Response: The sky appears blue because the shorter wavelength of blue light is scattered more by Earth's atmosphere due to its interaction with sunlight, making blue light spread out and appear when we look up at the sky.
--------------------------------------------------
Prompt: What is a metric ton?
Base Response: A metric ton, also known simply as a ton in some contexts, is a unit of mass equal to 1,000 kilograms or approximately 2,204.6 pounds. It's equivalent to one megagram (Mg). This unit is commonly used in many countries for measuring the weight of heavy objects and cargo.

In everyday use, especially in international trade and commerce, it is often referred to as "ton" without specifying "metric," but it is important to note that there are different types of tons:

- **Long Ton**: Also known as the Imperial ton, this is primarily used in the United Kingdom and other Commonwealth countries. It is equivalent to 1

In [5]:
training_args = DPOConfig(
    output_dir="qwen2.5-3b-dpo-output",
    beta=0.1,
    max_length=1024,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    max_steps=120,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=(compute_dtype == torch.bfloat16),
    fp16=(compute_dtype == torch.float16),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=20,
    save_steps=20,
    report_to="none",
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=train_mapped,
    eval_dataset=eval_mapped,
    processing_class=tokenizer,
)

model.config.use_cache = False

trainer.train()

model.save_pretrained("qwen2.5-3b-dpo-adapter")
tokenizer.save_pretrained("qwen2.5-3b-dpo-adapter")

print("DPO adapter successfully saved!")

{'loss': '0.6744', 'rewards/accuracies': '0.5375', 'rewards/margins': '0.04118'}
{'loss': '0.6026', 'rewards/accuracies': '0.7750', 'rewards/margins': '0.3985'}
{'eval_loss': '0.3898', 'eval_rewards/accuracies': '0.8929', 'eval_rewards/margins': '1.557'}
{'loss': '0.4785', 'rewards/accuracies': '0.8125', 'rewards/margins': '1.560'}
{'loss': '0.3889', 'rewards/accuracies': '0.7838', 'rewards/margins': '2.505'}
{'eval_loss': '0.4054', 'eval_rewards/accuracies': '0.9107', 'eval_rewards/margins': '3.045'}
{'loss': '0.3601', 'rewards/accuracies': '0.8375', 'rewards/margins': '2.277'}
{'loss': '0.1947', 'rewards/accuracies': '0.9625', 'rewards/margins': '2.791'}
{'eval_loss': '0.4325', 'eval_rewards/accuracies': '0.8750', 'eval_rewards/margins': '1.973'}
{'loss': '0.2091', 'rewards/accuracies': '0.9730', 'rewards/margins': '2.637'}
{'loss': '0.1322', 'rewards/accuracies': '0.9875', 'rewards/margins': '2.848'}
{'eval_loss': '0.4177', 'eval_rewards/accuracies': '0.8750', 'eval_rewards/margins'

In [6]:
for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            pad_token_id=tokenizer.pad_token_id,
        )
    aligned_response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    ).strip()

    print(f"Prompt: {prompt}")
    print(f"\033[91mBefore DPO (Base Model):\033[0m")
    print(base_responses[prompt])
    print(f"\033[92mAfter DPO (Aligned Model):\033[0m")
    print(aligned_response)
    print("-" * 80 + "\n")

Prompt: Explain why the sky is blue in one concise sentence.
Before DPO (Base Model):
The sky appears blue because the shorter wavelength of blue light is scattered more by Earth's atmosphere due to its interaction with sunlight, making blue light spread out and appear when we look up at the sky.
After DPO (Aligned Model):
The sky appears blue because of the way sunlight interacts with Earth's atmosphere; short wavelengths like blue light are scattered more by air molecules as it travels from space to our eyes, making the sky look blue.
--------------------------------------------------------------------------------

Prompt: What is a metric ton?
Before DPO (Base Model):
A metric ton, also known simply as a ton in some contexts, is a unit of mass equal to 1,000 kilograms or approximately 2,204.6 pounds. It's equivalent to one megagram (Mg). This unit is commonly used in many countries for measuring the weight of heavy objects and cargo.

In everyday use, especially in international tra